# Gold Layer — dim_products
Build product dimension from silver CRM + ERP tables.

## Setup Connection

In [ ]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session
from clickzetta.zettapark import functions as F
from clickzetta.zettapark.window import Window

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    "gold",
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]

## The Transformation Logic

In [ ]:
pn = session.table(f"silver.crm_products")
pc = session.table(f"silver.erp_product_category")

joined = pn.join(pc, pn["category_id"] == pc["category_id"], "left")

df = joined.select(
    pn["product_id"].alias("product_id"),
    pn["product_number"].alias("product_number"),
    pn["product_name"].alias("product_name"),
    pn["category_id"].alias("category_id"),
    pc["category"].alias("category"),
    pc["subcategory"].alias("subcategory"),
    pc["maintenance_flag"].alias("maintenance_flag"),
    pn["product_line"].alias("product_line"),
    pn["start_date"].alias("start_date"),
)

w = Window.order_by(F.col("start_date"), F.col("product_number"))
df = df.with_column("product_key", F.row_number().over(w))
df = df.select(
    "product_key", "product_id", "product_number", "product_name",
    "category_id", "category", "subcategory", "maintenance_flag",
    "product_line", "start_date",
)

## Sanity Check

In [ ]:
df.limit(10).show()

## Write Gold Table

In [ ]:
df.write.save_as_table(f"gold.dim_products", mode="overwrite")
print("dim_products OK")

## Verify

In [ ]:
session.table(f"gold.dim_products").limit(5).show()